# 04 Quality Checks

- esegue controlli ripetibili sul primo mart dichiarato in config
- usa le chiavi di validazione del mart quando disponibili
- aiuta a investigare anomalie residue dopo `toolkit validate all`

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import duckdb
import yaml

ROOT = Path('.').resolve()
DATASET_YML = (ROOT / 'dataset.yml').resolve() if (ROOT / 'dataset.yml').exists() else (ROOT / '..' / 'dataset.yml').resolve()
CFG = yaml.safe_load(DATASET_YML.read_text(encoding='utf-8'))
DATASET = CFG['dataset']['name']
YEARS = CFG['dataset']['years']
YEAR_INDEX = 0
YEAR = YEARS[YEAR_INDEX] if YEARS and 0 <= YEAR_INDEX < len(YEARS) else YEARS[0]
TABLES = CFG.get('mart', {}).get('tables', [])
TABLE_INDEX = 0
SELECTED_TABLE = TABLES[TABLE_INDEX] if TABLES and 0 <= TABLE_INDEX < len(TABLES) else (TABLES[0] if TABLES else {'name': 'mart_ok'})
TABLE_NAME = SELECTED_TABLE['name']
TABLE_RULES = CFG.get('mart', {}).get('validate', {}).get('table_rules', {}).get(TABLE_NAME, {})
KEY_COLUMNS = TABLE_RULES.get('primary_key', [])
CLI_PREFIX = ['toolkit'] if shutil.which('toolkit') else ['py', '-m', 'toolkit.cli.app']
INSPECT_CMD = CLI_PREFIX + ['inspect', 'paths', '--config', str(DATASET_YML), '--year', str(YEAR), '--json']
INSPECT = json.loads(subprocess.run(INSPECT_CMD, capture_output=True, text=True, check=True).stdout)
MART_OUTPUTS = INSPECT['paths']['mart']['outputs']
MART_PATH = Path(MART_OUTPUTS[TABLE_INDEX]) if 0 <= TABLE_INDEX < len(MART_OUTPUTS) else None
{'YEARS': YEARS, 'YEAR_INDEX': YEAR_INDEX, 'TABLES': [table['name'] for table in TABLES], 'TABLE_INDEX': TABLE_INDEX, 'TABLE_NAME': TABLE_NAME, 'MART_PATH': str(MART_PATH), 'INSPECT_CMD': INSPECT_CMD}

In [ ]:
con = duckdb.connect()
NUMERIC_COLUMNS = []

def detect_numeric_columns(path):
    rows = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path.as_posix()}')").fetchall()
    return [row[0] for row in rows if any(token in str(row[1]).upper() for token in ['INT', 'DECIMAL', 'DOUBLE', 'FLOAT', 'REAL', 'BIGINT'])]

if MART_PATH and MART_PATH.exists():
    NUMERIC_COLUMNS = detect_numeric_columns(MART_PATH)[:3]
    print({'KEY_COLUMNS': KEY_COLUMNS, 'NUMERIC_COLUMNS': NUMERIC_COLUMNS})
else:
    print('MART parquet not found.')

In [ ]:
if MART_PATH and MART_PATH.exists():
    if KEY_COLUMNS:
        keys = ', '.join(KEY_COLUMNS)
        dup_df = con.execute(
            f"SELECT {keys}, COUNT(*) AS dup_count FROM read_parquet('{MART_PATH.as_posix()}') GROUP BY {keys} HAVING COUNT(*) > 1 ORDER BY dup_count DESC LIMIT 20"
        ).df()
        display(dup_df)
    else:
        print('Duplicate-key check skipped: no primary key declared for this mart.')

    columns = [row[0] for row in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{MART_PATH.as_posix()}')").fetchall()]
    null_expr = ', '.join([f"AVG(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS {col}_null_rate" for col in columns])
    null_df = con.execute(f"SELECT {null_expr} FROM read_parquet('{MART_PATH.as_posix()}')").df().T.reset_index()
    display(null_df)

    if NUMERIC_COLUMNS:
        range_expr = ', '.join([f"MIN({col}) AS {col}_min, MAX({col}) AS {col}_max" for col in NUMERIC_COLUMNS])
        range_df = con.execute(f"SELECT {range_expr} FROM read_parquet('{MART_PATH.as_posix()}')").df().T.reset_index()
        display(range_df)
else:
    print('No MART output available.')